In [244]:
import pandas as pd
import numpy as np

In [245]:
df = pd.read_csv('data_set_da_test.csv')
print(df.shape)
df.head(10)

(637238, 6)


,event_date,session,user,page_type,event_type,product
0,2022-10-08 17:02:41,14274187577460658115s,2006979063809820329u,search_listing_page,page_view,0
1,2022-10-08 17:06:19,14274187577460658115s,2006979063809820329u,search_listing_page,page_view,0
2,2022-10-08 22:19:47,2704204808571844605s,2007646148110679693u,listing_page,page_view,0
3,2022-10-08 22:24:30,8970170322512311099s,11839491588321754710u,search_listing_page,page_view,0
4,2022-10-08 21:22:20,16223970371660715740s,11839887495958431209u,product_page,page_view,0
5,2022-10-08 09:32:33,18071599838924847030s,11840488544463900252u,product_page,page_view,0
6,2022-10-03 13:55:11,14766241913479479142s,2465777020541099786u,listing_page,page_view,0
7,2022-10-11 21:37:30,14295968418153070037s,17910835123677111857u,product_page,page_view,0
8,2022-10-11 19:34:59,12229440244945054666s,17912676908806342770u,product_page,page_view,0
9,2022-10-02 08:37:45,18126509212678608005s,3471971272464343927u,product_page,page_view,0


## Memory Saving

In [246]:
memory_before = df.memory_usage(deep=True).sum()

df["event_date"] = pd.to_datetime(df["event_date"])

for col in df.select_dtypes(include="integer").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

for col in df.select_dtypes(include="object").columns:
    categorical = df[col].astype("category")
    if categorical.memory_usage(deep=True) < df[col].memory_usage(deep=True):
        df[col] = categorical

memory_after = df.memory_usage(deep=True).sum()

print(f"Before: {memory_before / 1024**2:.1f} MiB")
print(f"After:  {memory_after / 1024**2:.1f} MiB")
print(f"Saved:  {(memory_before - memory_after) / 1024**2:.1f} MiB")
print(df.dtypes)

/var/folders/9l/mynyqp_x1qx1lcr2_jgyf1b40000gn/T/ipykernel_3962/2944869872.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Before: 79.0 MiB
After:  30.5 MiB
Saved:  48.5 MiB
event_date    datetime64[us]
session             category
user                category
page_type           category
event_type          category
product                int32
dtype: object


## Listing Different Possibilities

In [247]:
pages = df['page_type'].drop_duplicates().to_frame()
events = df['event_type'].drop_duplicates().to_frame()
pages_events = df[['page_type','event_type']].drop_duplicates()

display(pages)
display(events)
display(pages_events)

,page_type
0,search_listing_page
2,listing_page
4,product_page
166,order_page


,event_type
0,page_view
27,add_to_cart
166,order


,page_type,event_type
0,search_listing_page,page_view
2,listing_page,page_view
4,product_page,page_view
27,product_page,add_to_cart
166,order_page,order
248,search_listing_page,add_to_cart
947,listing_page,add_to_cart


## Sessions with orders

In [248]:
session_ids_test = df['session'].drop_duplicates()[:20]
session_ids_test_order = df[df['event_type']=='order']['session'].drop_duplicates().to_list()[:2]

if type(session_ids_test_order)==list:
    display(df[df['session'].isin(session_ids_test_order)].sort_values(['session','event_date']))
else: 
    display(df[df['session']==session_ids_test_order].sort_values(['session','event_date']))

,event_date,session,user,page_type,event_type,product
299252,2022-10-10 10:42:10,13850119679929065982s,8733494140597970234u,product_page,page_view,0
387846,2022-10-10 10:43:51,13850119679929065982s,8733494140597970234u,listing_page,page_view,0
481622,2022-10-10 10:43:51,13850119679929065982s,8733494140597970234u,listing_page,page_view,0
166,2022-10-10 10:44:00,13850119679929065982s,8733494140597970234u,order_page,order,0
530219,2022-10-06 16:29:20,16351345959988978592s,7011174022396541275u,search_listing_page,page_view,0
530216,2022-10-06 16:29:32,16351345959988978592s,7011174022396541275u,product_page,page_view,0
530215,2022-10-06 16:30:01,16351345959988978592s,7011174022396541275u,search_listing_page,page_view,0
530218,2022-10-06 16:30:07,16351345959988978592s,7011174022396541275u,product_page,page_view,0
573997,2022-10-06 16:30:28,16351345959988978592s,7011174022396541275u,product_page,page_view,0
551936,2022-10-06 16:30:40,16351345959988978592s,7011174022396541275u,search_listing_page,page_view,0


## Users with order in a single access

In [249]:
users_count = (
    df
    .groupby("user")
    .agg(
        distinct_sessions=("session", "nunique"),
        row_count=("event_type", 'size'),
    ).reset_index()
)

users_with_orders = (
    df[df['event_type']=='order']['user'].drop_duplicates().to_list()
)

users_only_order = (
    users_count[
        (users_count['user'].isin(users_with_orders)) & 
        (users_count['distinct_sessions']==1) & 
        (users_count['row_count']==1)]['user']
        .to_list()
        [:10])

if type(users_only_order)==list:
    display(df[df['user'].isin(users_only_order)])
else:
    display(df[df['user']==users_only_order])

,event_date,session,user,page_type,event_type,product
83127,2022-10-12 10:48:31,2901808355814938469s,10140551729934258948u,order_page,order,0
147802,2022-10-13 14:28:43,1279444712013020792s,10302079708853371394u,order_page,order,0
194447,2022-10-01 17:51:44,8390462509465841266s,10273412998665143959u,order_page,order,0
218414,2022-10-11 06:18:40,16461625766241290619s,10101210107924203118u,order_page,order,0
220419,2022-10-11 17:44:31,10526305291801504415s,10193494007908747402u,order_page,order,0
225166,2022-10-11 10:33:59,18159899398452306707s,10328998011321602350u,order_page,order,0
258264,2022-10-06 10:38:02,14766671639807764226s,10049404185838316964u,order_page,order,0
340096,2022-10-11 08:17:43,17336833989704731753s,10288869050993134845u,order_page,order,0
580704,2022-10-03 11:55:37,815891256738293999s,10172530067137598810u,order_page,order,0
587977,2022-10-05 20:10:56,11414244361566454287s,10362019717652192098u,order_page,order,0


## Users Full Historical

In [250]:
users_with_order = (
    users_count[
        (users_count['user'].isin(users_with_orders)) & 
        (users_count['distinct_sessions']==2) & #filtering users with a single session
        (users_count['row_count']>5)]['user'] #filtering by minimum number of rows
        .to_list()
        [1])

if type(users_with_order)==list:
    display(df[df['user'].isin(users_with_order)].sort_values(['event_date']))
else:
    display(df[df['user']==users_with_order].sort_values(['event_date']))

,event_date,session,user,page_type,event_type,product
373729,2022-10-04 09:59:30,10267854726623112600s,10045509651238878959u,listing_page,page_view,0
158160,2022-10-04 09:59:36,10267854726623112600s,10045509651238878959u,listing_page,page_view,0
410833,2022-10-04 10:01:08,10267854726623112600s,10045509651238878959u,product_page,page_view,0
158163,2022-10-04 10:01:49,10267854726623112600s,10045509651238878959u,product_page,add_to_cart,20755395
410832,2022-10-04 10:10:47,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0
410831,2022-10-04 10:10:52,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0
373730,2022-10-04 10:11:28,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0
158161,2022-10-04 10:11:30,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0
158162,2022-10-04 10:11:34,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0
410830,2022-10-04 10:11:44,10267854726623112600s,10045509651238878959u,search_listing_page,page_view,0


## Session Summary (original df)

In [251]:
event_counts = (
    df.groupby(["session", "event_type"])
    .size()
    .unstack(fill_value=0)
    .rename_axis(columns=None)
    .reset_index()
)

first_page_type = (
    df.sort_values("event_date", kind="stable")
    .groupby("session")["page_type"]
    .first()
    .rename("first_page_type")
    .reset_index()
)

session_summary = (
    df.groupby("session", as_index=False)
    .agg(
        first_event_timestamp=("event_date", "min"),
        latest_event_timestamp=("event_date", "max"),
        product_count=(
            "product",
            lambda products: (
                products.ne(0)
            ).sum(),
        ),
    )
    .merge(event_counts, on="session", how="left")
    .merge(first_page_type, on="session", how="left")
)

session_summary = session_summary.copy()[['session', 'first_event_timestamp', 'first_page_type','first_event_timestamp', 'product_count', 'page_view', 'add_to_cart', 'order']]

session_summary.head()

,session,first_event_timestamp,first_page_type,first_event_timestamp,product_count,page_view,add_to_cart,order
0,10000000052138714153s,2022-10-12 16:26:46,search_listing_page,2022-10-12 16:26:46,0,1,0,0
1,10000003251787879077s,2022-10-01 19:16:18,search_listing_page,2022-10-01 19:16:18,0,2,0,0
2,10000028460721229362s,2022-10-13 09:08:27,listing_page,2022-10-13 09:08:27,0,2,0,0
3,10000145746398845278s,2022-10-04 12:59:05,listing_page,2022-10-04 12:59:05,0,1,0,0
4,1000015740551295390s,2022-10-03 16:03:53,listing_page,2022-10-03 16:03:53,0,1,0,0


## Unified Sessions

In [252]:
max_days_between_sessions = 5
max_session_gap = pd.Timedelta(days=max_days_between_sessions)

session_info = (
    df.groupby(["user", "session"], as_index=False)
    .agg(
        session_start=("event_date", "min"),
        session_end=("event_date", "max"),
        has_order=("event_type", lambda event_types: event_types.eq("order").any()),
    )
    .sort_values(["user", "session_start", "session_end", "session"])
)

sessions_by_user = session_info.groupby("user", sort=False)
session_info["next_session"] = sessions_by_user["session"].shift(-1)
session_info["next_session_start"] = sessions_by_user["session_start"].shift(-1)
session_info["next_has_order"] = sessions_by_user["has_order"].shift(-1).eq(True)
session_gap = session_info["next_session_start"] - session_info["session_end"]

sessions_to_merge = session_info.loc[
    ~session_info["has_order"]
    & session_info["next_has_order"]
    & session_gap.ge(pd.Timedelta(0))
    & session_gap.le(max_session_gap),
    ["user", "session", "next_session"],
]

session_mapping = {
    (row.user, row.next_session): row.session
    for row in sessions_to_merge.itertuples(index=False)
}
row_session_keys = pd.MultiIndex.from_arrays([df["user"], df["session"]])
df_unified = df.copy()
df_unified["unified_session"] = pd.Series(
    row_session_keys.map(session_mapping), index=df.index
).fillna(df["session"])
df_unified = df_unified.copy().sort_values(['unified_session', 'event_date'])

print(f"Unified {len(sessions_to_merge)} session pairs.", '\n')
print(f'df and df_unified have the same row numbers: {df.shape[0]==df_unified.shape[0]}')

df_unified.head()

Unified 1604 session pairs. 

df and df_unified have the same row numbers: True


,event_date,session,user,page_type,event_type,product,unified_session
449481,2022-10-12 16:26:46,10000000052138714153s,8035194775607769271u,search_listing_page,page_view,0,10000000052138714153s
293523,2022-10-01 19:16:18,10000003251787879077s,2613381957342920411u,search_listing_page,page_view,0,10000003251787879077s
66383,2022-10-01 19:16:56,10000003251787879077s,2613381957342920411u,search_listing_page,page_view,0,10000003251787879077s
172709,2022-10-13 09:08:27,10000028460721229362s,12451948957440700749u,listing_page,page_view,0,10000028460721229362s
508640,2022-10-13 09:08:33,10000028460721229362s,12451948957440700749u,listing_page,page_view,0,10000028460721229362s


### Testing unification
User ID with multiple sessions: 10045509651238878959u;
Unified Session ID: 10267854726623112600s.

In [253]:
df_unified[df_unified['unified_session'].eq('10000003251787879077s')]

,event_date,session,user,page_type,event_type,product,unified_session
293523,2022-10-01 19:16:18,10000003251787879077s,2613381957342920411u,search_listing_page,page_view,0,10000003251787879077s
66383,2022-10-01 19:16:56,10000003251787879077s,2613381957342920411u,search_listing_page,page_view,0,10000003251787879077s


### Session Summary (unified df)

In [254]:
event_counts_unified = (
    df_unified.groupby(["unified_session", "event_type"])
    .size()
    .unstack(fill_value=0)
    .rename_axis(columns=None)
    .reset_index()
)

first_page_type_unified = (
    df_unified.sort_values("event_date", kind="stable")
    .groupby("unified_session")["page_type"]
    .first()
    .rename("first_page_type")
    .reset_index()
)

session_summary_unified = (
    df_unified.groupby("unified_session", as_index=False)
    .agg(
        first_event_timestamp=("event_date", "min"),
        latest_event_timestamp=("event_date", "max"),
        product_count=(
            "product",
            lambda products: (
                products.ne(0)
            ).sum(),
        ),
    )
    .merge(event_counts_unified, on="unified_session", how="left")
    .merge(first_page_type_unified, on="unified_session", how="left")
)

session_summary_unified['session_duration_h'] = (
    session_summary_unified['latest_event_timestamp']
    - session_summary_unified['first_event_timestamp']
).dt.total_seconds() / 3600

session_summary_unified = session_summary_unified.copy()[['unified_session', 'first_page_type','first_event_timestamp','latest_event_timestamp', 'session_duration_h','product_count', 'page_view', 'add_to_cart', 'order']]

session_summary_unified.head()

,unified_session,first_page_type,first_event_timestamp,latest_event_timestamp,session_duration_h,product_count,page_view,add_to_cart,order
0,10000000052138714153s,search_listing_page,2022-10-12 16:26:46,2022-10-12 16:26:46,0.000000,0,1,0,0
1,10000003251787879077s,search_listing_page,2022-10-01 19:16:18,2022-10-01 19:16:56,0.010556,0,2,0,0
2,10000028460721229362s,listing_page,2022-10-13 09:08:27,2022-10-13 09:08:33,0.001667,0,2,0,0
3,10000145746398845278s,listing_page,2022-10-04 12:59:05,2022-10-04 12:59:05,0.000000,0,1,0,0
4,1000015740551295390s,listing_page,2022-10-03 16:03:53,2022-10-03 16:03:53,0.000000,0,1,0,0


### Testing one of the unified sessions

In [255]:
sessions_by_unified = df_unified.groupby('unified_session')['session'].nunique().rename('n_sessions').reset_index().sort_values('n_sessions', ascending=False)
sessions_by_unified

,unified_session,n_sessions
296372,7914552649112004363s,2
256872,5976353290692264275s,2
51557,12512840233644949114s,2
267210,6484803917862912628s,2
314498,8803718605114740322s,2
...,...,...
113215,15536051841897719191s,1
113214,15536009992612565199s,1
113213,15535944249747469048s,1
113212,15535874281976007538s,1


In [256]:
unified_session_id_test = '7914552649112004363s'

display(session_summary_unified[session_summary_unified['unified_session']==unified_session_id_test])

display(df_unified[df_unified['unified_session'].eq(unified_session_id_test)])

,unified_session,first_page_type,first_event_timestamp,latest_event_timestamp,session_duration_h,product_count,page_view,add_to_cart,order
296372,7914552649112004363s,product_page,2022-10-03 17:30:39,2022-10-03 18:39:41,1.150556,1,5,1,1


,event_date,session,user,page_type,event_type,product,unified_session
85360,2022-10-03 17:30:39,7914552649112004363s,11912537919138244027u,product_page,page_view,0,7914552649112004363s
85361,2022-10-03 17:32:35,7914552649112004363s,11912537919138244027u,product_page,page_view,0,7914552649112004363s
275535,2022-10-03 18:09:01,17352327659053400306s,11912537919138244027u,product_page,page_view,0,7914552649112004363s
142904,2022-10-03 18:10:40,17352327659053400306s,11912537919138244027u,product_page,page_view,0,7914552649112004363s
275536,2022-10-03 18:10:42,17352327659053400306s,11912537919138244027u,product_page,add_to_cart,29504782,7914552649112004363s
85362,2022-10-03 18:34:48,17352327659053400306s,11912537919138244027u,product_page,page_view,0,7914552649112004363s
275537,2022-10-03 18:39:41,17352327659053400306s,11912537919138244027u,order_page,order,0,7914552649112004363s


In [257]:
(
    df_unified
    .groupby(['page_type', 'event_type'])
    [['page_type', 'event_type']]
    .size()
    .rename('n_interactions')
    .reset_index()
    .sort_values('n_interactions', ascending=False)
    )

,page_type,event_type,n_interactions
4,product_page,page_view,270750
1,listing_page,page_view,231263
6,search_listing_page,page_view,110485
3,product_page,add_to_cart,12200
2,order_page,order,8741
5,search_listing_page,add_to_cart,3273
0,listing_page,add_to_cart,526


In [258]:
session_summary_unified[session_summary_unified['session_duration_h']==max(session_summary_unified['session_duration_h'])]

,unified_session,first_page_type,first_event_timestamp,latest_event_timestamp,session_duration_h,product_count,page_view,add_to_cart,order
10282,10495458049874864571s,listing_page,2022-10-05 22:24:35,2022-10-10 23:13:26,120.814167,1,9,1,1
